# Execution Layer Function Smoke Test

This notebook checks the execution layer command surface with the dry-run/mock backend. It does not require a real robot, CAN device, gripper, camera, or separate execution process.

Covered command types:

- `CONNECT`, `DISCONNECT`, `ENABLE`, `STOP`, `RESET`, `GO_HOME`
- `GET_STATE`, `GET_TCP_POSE`, `GET_JOINTS`
- `MOVE_J`, `MOVE_P`, `SERVO_DELTA_POSE`
- `OPEN_GRIPPER`, `CLOSE_GRIPPER`

Use this as a fast API availability check before running hardware-level validation.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src" / "wbcd_task1").exists():
    ROOT = Path.cwd().parents[0]

for path in (ROOT / "src", ROOT):
    text = str(path)
    if text not in sys.path:
        sys.path.insert(0, text)

ROOT

In [ ]:
from wbcd_task1.core.config import load_task_config
from wbcd_task1.execution.server.agilex_server import AgilexExecutionServer, build_dispatcher
from wbcd_task1.execution.server.command_types import ExecutionCommand, ExecutionCommandType

config = load_task_config(ROOT / "configs" / "debug_execution.yaml", dry_run=True)
dispatcher = build_dispatcher(config, dry_run=True)
ARM = "right"

def run(command_type, arm=None, **payload):
    command = ExecutionCommand(command_type=command_type, arm=arm, payload=payload)
    response = dispatcher.dispatch(command)
    assert response.ok, f"{command_type.value} failed: {response.message}"
    return response.data

def arm_state(arm=ARM):
    return run(ExecutionCommandType.GET_STATE, arm=arm)["state"][arm]

print("Execution dispatcher ready")

## Connection and State Commands

In [ ]:
run(ExecutionCommandType.CONNECT)
run(ExecutionCommandType.ENABLE, arm=ARM)

state = arm_state()
assert state["connected"] is True
assert state["enabled"] is True
assert len(state["joints"]) == 7
assert len(state["tcp_pose"]) == 6

joints = run(ExecutionCommandType.GET_JOINTS, arm=ARM)["joints"]
tcp_pose = run(ExecutionCommandType.GET_TCP_POSE, arm=ARM)["tcp_pose"]
assert len(joints) == 7
assert len(tcp_pose) == 6

state

## Joint Motion

In [ ]:
target_joints = [0.1, -0.2, 0.1, 1.7, 0.5, 0.1, 1.0]
run(ExecutionCommandType.MOVE_J, arm=ARM, joints=target_joints, delta=False)
actual_joints = run(ExecutionCommandType.GET_JOINTS, arm=ARM)["joints"]
assert actual_joints == target_joints

delta_joints = [0.01, 0.0, 0.0, 0.0, 0.0, 0.0, -0.01]
run(ExecutionCommandType.MOVE_J, arm=ARM, joints=delta_joints, delta=True)
actual_joints = run(ExecutionCommandType.GET_JOINTS, arm=ARM)["joints"]
assert round(actual_joints[0], 6) == round(target_joints[0] + delta_joints[0], 6)
assert round(actual_joints[-1], 6) == round(target_joints[-1] + delta_joints[-1], 6)

actual_joints

## Cartesian Motion

In [ ]:
target_pose = [0.30, -0.05, 0.25, 0.0, 0.0, 0.0]
run(ExecutionCommandType.MOVE_P, arm=ARM, pose=target_pose, delta=False)
actual_pose = run(ExecutionCommandType.GET_TCP_POSE, arm=ARM)["tcp_pose"]
assert actual_pose == target_pose

delta_pose = [0.01, 0.0, 0.0, 0.0, 0.0, 0.0]
run(ExecutionCommandType.MOVE_P, arm=ARM, pose=delta_pose, delta=True)
actual_pose = run(ExecutionCommandType.GET_TCP_POSE, arm=ARM)["tcp_pose"]
assert round(actual_pose[0], 6) == round(target_pose[0] + delta_pose[0], 6)

actual_pose

## Servo Delta Pose and Safety Clamp

`debug_execution.yaml` sets `max_linear_step_m=0.005` and `max_angular_step_rad=0.05`, so a larger servo command should be clamped by `SafetyGuard`.

In [ ]:
large_delta = [0.2, -0.2, 0.2, 0.5, -0.5, 0.5]
data = run(ExecutionCommandType.SERVO_DELTA_POSE, arm=ARM, delta_pose=large_delta)
clamped = data["delta_pose"]

assert clamped == [0.005, -0.005, 0.005, 0.05, -0.05, 0.05]
clamped

## Gripper Commands

In [ ]:
run(ExecutionCommandType.OPEN_GRIPPER, arm=ARM, width_m=0.08, force=1.0)
state = arm_state()
assert state["gripper_width_m"] == 0.08

run(ExecutionCommandType.CLOSE_GRIPPER, arm=ARM, width_m=0.0, force=1.0)
state = arm_state()
assert state["gripper_width_m"] == 0.0

state

## Reset, Home, Stop, and Disconnect

In [ ]:
run(ExecutionCommandType.RESET, arm=ARM)
run(ExecutionCommandType.GO_HOME, arm=ARM)
state = arm_state()
assert state["tcp_pose"] == [0.3, 0.0, 0.25, 0.0, 0.0, 0.0]

run(ExecutionCommandType.STOP, arm=ARM)
run(ExecutionCommandType.DISCONNECT)
state = arm_state()
assert state["connected"] is False
assert state["enabled"] is False

state

## Migrated Nero Server Backend Mapping

This section uses a fake `NeroDualArmServer` with the same old method names. It verifies that the real backend maps task-level execution commands to the migrated `nero_interface_server.py` methods rather than bypassing them.

In [ ]:
from wbcd_task1.execution.sdk.nero_interface_backend import NeroInterfaceServerBackend
from wbcd_task1.execution.server.command_dispatcher import CommandDispatcher
from wbcd_task1.execution.server.safety_guard import SafetyGuard, SafetyLimits

class FakeRobot:
    def __init__(self, calls):
        self.calls = calls
    def reset(self):
        self.calls.append(("robot.reset",))

class FakeNeroDualArmServer:
    def __init__(self, gripper_enabled=True, tcp_offset_enabled=False, limit_z=0.07):
        self.calls = []
        self.right_robot = FakeRobot(self.calls)
        self.calls.append(("__init__", gripper_enabled, tcp_offset_enabled, limit_z))
    def right_robot_get_joint_positions(self):
        self.calls.append(("right_robot_get_joint_positions",))
        return [0.0] * 7
    def right_robot_get_ee_pose(self):
        self.calls.append(("right_robot_get_ee_pose",))
        return [0.0] * 6
    def right_robot_go_home(self):
        self.calls.append(("right_robot_go_home",))
    def right_robot_move_to_joint_positions(self, joints, delta=False):
        self.calls.append(("right_robot_move_to_joint_positions", joints, delta))
    def right_robot_move_to_ee_pose(self, pose, delta=False):
        self.calls.append(("right_robot_move_to_ee_pose", pose, delta))
    def servo_p_OL(self, robot_arm, pose, delta=False):
        self.calls.append(("servo_p_OL", robot_arm, pose, delta))
        return True
    def right_gripper_goto(self, width, force=1.0):
        self.calls.append(("right_gripper_goto", width, force))
        return True
    def right_gripper_get_state(self):
        self.calls.append(("right_gripper_get_state",))
        return {"width": 0.04}
    def robot_stop(self, robot_arm):
        self.calls.append(("robot_stop", robot_arm))
        return True

server_backend = NeroInterfaceServerBackend(
    arms=[ARM],
    enable_gripper=True,
    limit_z=0.0,
    server_cls=FakeNeroDualArmServer,
)
server_dispatcher = CommandDispatcher(
    backend=server_backend,
    safety_guard=SafetyGuard(SafetyLimits(min_tcp_z_m=0.0, max_servo_rate_hz=0.0)),
)

def server_run(command_type, arm=None, **payload):
    response = server_dispatcher.dispatch(ExecutionCommand(command_type, arm=arm, payload=payload))
    assert response.ok, response.message
    return response.data

server_run(ExecutionCommandType.CONNECT)
server_run(ExecutionCommandType.MOVE_J, arm=ARM, joints=[0.1] * 7)
server_run(ExecutionCommandType.MOVE_P, arm=ARM, pose=[0.1, 0.0, 0.2, 0.0, 0.0, 0.0])
server_run(ExecutionCommandType.SERVO_DELTA_POSE, arm=ARM, delta_pose=[0.001, 0, 0, 0, 0, 0])
server_run(ExecutionCommandType.OPEN_GRIPPER, arm=ARM, width_m=0.08, force=1.0)
server_run(ExecutionCommandType.CLOSE_GRIPPER, arm=ARM, width_m=0.0, force=1.0)
server_run(ExecutionCommandType.RESET, arm=ARM)
server_run(ExecutionCommandType.GO_HOME, arm=ARM)
server_run(ExecutionCommandType.STOP, arm=ARM)

called = [call[0] for call in server_backend._server.calls]
assert "right_robot_move_to_joint_positions" in called
assert "right_robot_move_to_ee_pose" in called
assert "servo_p_OL" in called
assert called.count("right_gripper_goto") == 2
assert "robot.reset" in called
assert "right_robot_go_home" in called
assert "robot_stop" in called

server_backend._server.calls

## Local Execution Service Facade Smoke Test

This validates the `AgilexExecutionServer` local facade. No network server is started in this architecture.

In [ ]:
server = AgilexExecutionServer(config=config, dry_run=True)

assert server.connect()["ok"] is True
assert server.dispatch("enable", arm=ARM)["ok"] is True
assert server.go_home(arm=ARM)["ok"] is True
assert server.get_state(arm=ARM)["ok"] is True
assert server.dispatch("get_joints", arm=ARM)["ok"] is True
assert server.dispatch("get_tcp_pose", arm=ARM)["ok"] is True
assert server.dispatch("open_gripper", arm=ARM, payload={"width_m": 0.07})["ok"] is True
assert server.dispatch("close_gripper", arm=ARM, payload={"width_m": 0.0})["ok"] is True
assert server.stop(arm=ARM)["ok"] is True

print("local execution facade smoke test passed")

## Hardware Validation Notes

The cells above intentionally use dry-run. For real hardware validation, initialize the local service first:

```bash
python scripts/start_execution_server.py --config configs/wbcd_task1_tube_insert.yaml
python scripts/execution_smoke_test.py --config configs/debug_execution.yaml
```

Before removing dry-run, verify CAN names, robot clearance, gripper clearance, speed limits, `min_tcp_z_m`, and emergency stop readiness.